In [10]:
# Step 1: Install dependencies & set Hugging Face token
%pip install -q "datasets>=2.19.0" "huggingface_hub>=0.24"
import os
import getpass

# 直接设置Hugging Face token，跳过登录界面
hf_token = getpass.getpass("Paste your Hugging Face token: ")
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token

print("Hugging Face token set successfully!")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Paste your Hugging Face token:  ········


Hugging Face token set successfully!


In [11]:
# Step 2: Load FLARE-NER test set and normalize
from datasets import load_dataset, Dataset

# FLARE-NER dataset
ds_raw = load_dataset("ChanceFocus/flare-ner", split="test")
print("Loaded flare-ner test:", len(ds_raw), "columns:", ds_raw.column_names)

# Define entity types
ENTITY_TYPES = ["PER", "ORG", "LOC"]

def parse_ner_answer(answer_text):
    """Parse the answer format from flare-ner"""
    if not answer_text:
        return []
    
    entities = []
    lines = answer_text.strip().split('\n')
    for line in lines:
        if ',' in line:
            parts = line.rsplit(',', 1)
            if len(parts) == 2:
                entity_name = parts[0].strip()
                entity_type = parts[1].strip().upper()
                if entity_type in ENTITY_TYPES:
                    entities.append({"entity": entity_name, "type": entity_type})
    return entities

def _map_row(x):
    text = x.get("text") or x.get("sentence") or x.get("content") or ""
    answer = x.get("answer", "")
    entities = parse_ner_answer(answer)
    return {
        "text": text, 
        "answer": answer,
        "entities": entities,
        "entity_types": ENTITY_TYPES
    }

ds = Dataset.from_list([{**r, **_map_row(r)} for r in ds_raw])
bad = [i for i, r in enumerate(ds) if not r["text"]]
print("Samples with empty text:", len(bad))
assert len(bad) == 0, "Found samples with empty text."

Loaded flare-ner test: 98 columns: ['query', 'answer', 'label', 'text']
Samples with empty text: 0


In [12]:
# Step 3: Install dependencies, configure OpenAI, and record experiment metadata
%pip install -q "openai==1.40.2" "httpx==0.27.2" "httpcore==1.0.5" \
               "pandas>=2.2.2" "tqdm>=4.66.4" "requests>=2.31.0"

import os, getpass, json, time, platform
from importlib.metadata import version, PackageNotFoundError
import requests

# o3-mini配置
MODEL = "o3-mini"
BASE_URL = "https://api.openai.com/v1"

api_key = os.getenv("OPENAI_API_KEY") or os.getenv("API_KEY")
if not api_key:
    api_key = getpass.getpass("Paste your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

# 文件命名
run_tag = f"flare_ner_{MODEL.replace('-', '_')}"
save_dir = "/content"
pred_path = f"{save_dir}/{run_tag}_predictions.csv"
meta_path = f"{save_dir}/{run_tag}_metadata.json"

def ver(pkg: str) -> str:
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "not-installed"

meta = {
    "dataset": "ChanceFocus/flare-ner",
    "split": "test",
    "entity_types": list(ENTITY_TYPES),
    "model": MODEL,
    "openai_sdk": ver("openai"),
    "httpx": ver("httpx"),
    "httpcore": ver("httpcore"),
    "datasets_version": ver("datasets"),
    "pandas": ver("pandas"),
    "tqdm": ver("tqdm"),
    "time_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "python": platform.python_version(),
    "base_url": BASE_URL,
    "note": "o3-mini model evaluation on FLARE-NER"
}

os.makedirs(save_dir, exist_ok=True)
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Meta saved ->", meta_path)
print("MODEL:", MODEL, "| BASE_URL:", BASE_URL)
print("OPENAI_API_KEY is set:", bool(os.environ.get("OPENAI_API_KEY")))

Note: you may need to restart the kernel to use updated packages.
Meta saved -> /content/flare_ner_o3_mini_metadata.json
MODEL: o3-mini | BASE_URL: https://api.openai.com/v1
OPENAI_API_KEY is set: True



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Step 4: 强制重新运行（忽略之前的缓存）
import json, os, re, time
import pandas as pd
from tqdm import tqdm
import requests

# 重新定义文件路径（确保它们在当前作用域内）
run_tag = f"flare_ner_{MODEL.replace('-', '_')}"
save_dir = "/content"
pred_path = f"{save_dir}/{run_tag}_predictions.csv"
err_path = f"{save_dir}/{run_tag}_errors.csv"

print(f"预测文件路径: {pred_path}")
print(f"错误文件路径: {err_path}")

def _strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def _make_ner_prompt(text: str):
    return (
        "Extract named entities from the following financial text. "
        "Identify entities that represent a person ('PER'), an organization ('ORG'), or a location ('LOC').\n\n"
        f"Text: {text}\n\n"
        "Return ONLY the entities in this format:\n"
        "entity1, TYPE\n"
        "entity2, TYPE\n"
        "If no entities, return 'None'"
    )

def parse_entities_from_response(response_text: str):
    """Parse model response into list of entities"""
    if not response_text or response_text.strip().upper() == "NONE":
        return []
    
    entities = []
    lines = response_text.strip().split('\n')
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if ',' in line:
            parts = line.rsplit(',', 1)
            if len(parts) == 2:
                entity = parts[0].strip().strip('"').strip("'")
                etype = parts[1].strip().strip('"').strip("'").upper()
                if etype in ["PER", "ORG", "LOC"]:
                    entities.append({"entity": entity, "type": etype})
    return entities

def ask_o3_mini_once(text):
    """使用requests直接调用API，不带temperature参数"""
    url = f"{BASE_URL}/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    user_text = _make_ner_prompt(text)
    
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are a financial NLP expert. Respond only with the requested format."},
            {"role": "user", "content": user_text}
        ]
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        
        if response.status_code != 200:
            error_msg = f"API Error {response.status_code}: {response.text}"
            return [], "", False, error_msg
            
        result = response.json()
        
        if 'choices' not in result or len(result['choices']) == 0:
            return [], "", False, "No choices in response"
            
        txt = result['choices'][0]['message']['content']
        txt = _strip_code_fences(txt)
        entities = parse_entities_from_response(txt)
        return entities, txt, True, None
        
    except Exception as e:
        return [], "", False, str(e)

def ask_o3_mini(text):
    delay = 2.0
    for attempt in range(6):
        try:
            entities, raw_response, success, error = ask_o3_mini_once(text)
            if success:
                return entities, raw_response, True, None
            else:
                if attempt < 5:
                    time.sleep(delay)
                    delay = min(delay * 2, 60)
                    continue
                return [], raw_response, False, error
        except Exception as e:
            if attempt == 5:
                return [], "", False, str(e)
            time.sleep(delay)
            delay = min(delay * 2, 60)
    return [], "", False, "Exhausted retries"

# 先测试一个样本确认可以工作
print("测试单个样本...")
test_sample = ds[0]
test_text = test_sample["text"]
print(f"测试文本: {test_text[:100]}...")

test_entities, test_response, test_success, test_error = ask_o3_mini(test_text)
print(f"测试成功: {test_success}")
if test_success:
    print(f"识别的实体: {test_entities}")
    print(f"原始响应: {test_response}")
    
    print("\n开始完整评估（重新运行，忽略缓存）...")
    
    # 删除旧的预测文件以强制重新运行
    if os.path.exists(pred_path):
        os.remove(pred_path)
        print(f"已删除旧的预测文件: {pred_path}")
    if os.path.exists(err_path):
        os.remove(err_path)
        print(f"已删除旧的错误文件: {err_path}")
    
    rows_done = []
    done_idx = set()
    err_rows = []
    buf = []
    save_every = 10

    total = len(ds)
    print(f"Starting o3-mini NER evaluation on {total} samples...")

    for i in tqdm(range(total)):
        x = ds[i]
        text = x["text"]
        gold_entities = x["entities"]

        try:
            pred_entities, raw_response, success, error_msg = ask_o3_mini(text)
            if not success:
                raise RuntimeError(error_msg)
            raw = raw_response
        except Exception as e:
            pred_entities = []
            raw = f"ERROR: {type(e).__name__}: {e}"
            err_rows.append({"row_idx": i, "id": x.get("id", i), "error": raw, "text": text})
            success = False

        buf.append({
            "row_idx": i,
            "id": x.get("id", i),
            "text": text,
            "pred_raw": raw,
            "pred_entities": json.dumps(pred_entities),
            "gold_entities": json.dumps(gold_entities),
            "pred_count": len(pred_entities),
            "gold_count": len(gold_entities),
            "success": success
        })

        if len(buf) % save_every == 0:
            out = pd.DataFrame(rows_done + buf).sort_values("row_idx")
            out.to_csv(pred_path, index=False)
            if err_rows:
                pd.DataFrame(err_rows).to_csv(err_path, index=False)
            print(f"[checkpoint] saved {len(out)}/{total} -> {pred_path}")

    out = pd.DataFrame(rows_done + buf).sort_values("row_idx")
    out.to_csv(pred_path, index=False)
    if err_rows:
        pd.DataFrame(err_rows).to_csv(err_path, index=False)
    print(f"[done] o3-mini NER evaluation completed -> {pred_path}")
    
else:
    print(f"测试失败: {test_error}")

预测文件路径: /content/flare_ner_o3_mini_predictions.csv
错误文件路径: /content/flare_ner_o3_mini_errors.csv
测试单个样本...
测试文本: Subordinated Loan Agreement - Silicium de Provence SAS and Evergreen Solar Inc . 7 - December 2007 [...
测试成功: True
识别的实体: [{'entity': 'Silicium de Provence SAS', 'type': 'ORG'}, {'entity': 'Evergreen Solar Inc', 'type': 'ORG'}, {'entity': 'HERBERT SMITH', 'type': 'ORG'}]
原始响应: Silicium de Provence SAS, ORG
Evergreen Solar Inc, ORG
HERBERT SMITH, ORG

开始完整评估（重新运行，忽略缓存）...
已删除旧的错误文件: /content/flare_ner_o3_mini_errors.csv
Starting o3-mini NER evaluation on 98 samples...


 10%|████████▎                                                                         | 10/98 [01:16<11:50,  8.08s/it]

[checkpoint] saved 10/98 -> /content/flare_ner_o3_mini_predictions.csv


 20%|████████████████▋                                                                 | 20/98 [02:21<07:26,  5.72s/it]

[checkpoint] saved 20/98 -> /content/flare_ner_o3_mini_predictions.csv


...

 82%|██████████████████████████████████████████████████████████████████▉               | 80/98 [09:55<02:29,  8.28s/it]

[checkpoint] saved 80/98 -> /content/flare_ner_o3_mini_predictions.csv


 92%|███████████████████████████████████████████████████████████████████████████▎      | 90/98 [11:08<01:01,  7.72s/it]

[checkpoint] saved 90/98 -> /content/flare_ner_o3_mini_predictions.csv


 97%|███████████████████████████████████████████████████████████████████████████████▍  | 95/98 [11:30<00:14,  4.91s/it]

In [ ]:
# Step 5: Compute evaluation metrics for NER
%pip install -q scikit-learn

import pandas as pd
import json
import time

# 加载预测结果
if os.path.exists(pred_path):
    df = pd.read_csv(pred_path).sort_values("row_idx").drop_duplicates("row_idx", keep="last")

    # 解析实体
    def parse_entity_string(s):
        if pd.isna(s) or s == '[]' or s == '':
            return []
        try:
            return json.loads(s)
        except:
            return []

    df['gold_entities_parsed'] = df['gold_entities'].apply(parse_entity_string)
    df['pred_entities_parsed'] = df['pred_entities'].apply(parse_entity_string)

    # 只考虑成功的预测
    success_df = df[df['success'] == True].copy() if 'success' in df.columns else df.copy()
    print(f"Total samples: {len(df)}")
    print(f"Successful predictions: {len(success_df)}")
    print(f"Failed predictions: {len(df) - len(success_df)}")

    if len(success_df) > 0:
        # 计算实体级别的精确匹配
        exact_match_count = 0
        total_pred_entities = 0
        total_gold_entities = 0
        correct_predictions = 0
        correct_by_type = {"PER": 0, "ORG": 0, "LOC": 0}
        total_by_type = {"PER": 0, "ORG": 0, "LOC": 0}
        
        for _, row in success_df.iterrows():
            gold = row['gold_entities_parsed']
            pred = row['pred_entities_parsed']
            
            gold_set = {(e['entity'].lower(), e['type']) for e in gold}
            pred_set = {(e['entity'].lower(), e['type']) for e in pred}
            
            exact_match_count += 1 if gold_set == pred_set else 0
            
            total_gold_entities += len(gold)
            total_pred_entities += len(pred)
            
            correct_predictions += len(gold_set & pred_set)
            
            for e in gold:
                total_by_type[e['type']] = total_by_type.get(e['type'], 0) + 1
            
            for e in pred:
                if (e['entity'].lower(), e['type']) in gold_set:
                    correct_by_type[e['type']] = correct_by_type.get(e['type'], 0) + 1
        
        exact_match_accuracy = exact_match_count / len(success_df) if len(success_df) > 0 else 0
        precision = correct_predictions / total_pred_entities if total_pred_entities > 0 else 0
        recall = correct_predictions / total_gold_entities if total_gold_entities > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        print("\n" + "="*50)
        print("EVALUATION RESULTS - o3-mini on FLARE-NER")
        print("="*50)
        print(f"Exact Match Accuracy: {exact_match_accuracy:.4f}")
        print(f"\nEntity-Level Metrics (micro):")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1-Score: {f1:.4f}")
        
        print("\nPer-Entity Type Performance:")
        for etype in ["PER", "ORG", "LOC"]:
            if total_by_type[etype] > 0:
                type_recall = correct_by_type[etype] / total_by_type[etype] if total_by_type[etype] > 0 else 0
                print(f"  {etype}:")
                print(f"    Correct: {correct_by_type[etype]}/{total_by_type[etype]}")
                print(f"    Recall: {type_recall:.4f}")
        
        eval_results = {
            "model": MODEL,
            "dataset": "ChanceFocus/flare-ner",
            "split": "test",
            "total_samples": len(df),
            "successful_predictions": len(success_df),
            "failed_predictions": len(df) - len(success_df),
            "exact_match_accuracy": float(exact_match_accuracy),
            "entity_level_metrics": {
                "precision": float(precision),
                "recall": float(recall),
                "f1": float(f1),
                "correct_predictions": int(correct_predictions),
                "total_predicted": int(total_pred_entities),
                "total_gold": int(total_gold_entities)
            },
            "per_type_metrics": {
                etype: {
                    "correct": int(correct_by_type[etype]),
                    "total": int(total_by_type[etype]),
                    "recall": float(correct_by_type[etype] / total_by_type[etype]) if total_by_type[etype] > 0 else 0
                } for etype in ["PER", "ORG", "LOC"]
            },
            "evaluation_time": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())
        }
        
        eval_path = f"{save_dir}/{run_tag}_evaluation_results.json"
        with open(eval_path, "w") as f:
            json.dump(eval_results, f, indent=2)
        print(f"\nEvaluation results saved -> {eval_path}")
        
    else:
        print("No successful predictions to evaluate!")
else:
    print(f"预测文件不存在: {pred_path}")